In [21]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [22]:
!git clone https://github.com/deepakachu5114/ML-Project.git

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Cloning into 'ML-Project'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 48 (delta 14), reused 21 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 1.34 MiB | 15.40 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [23]:
%cd "/kaggle/working/ML-Project"

/kaggle/working/ML-Project


In [24]:
pip install sentence-transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [25]:
%%writefile preprocess_script.py
from preprocess import PreprocessForContextualEmbeddings
import pandas as pd
from constants import SAVEPATH

# Step 1: Preprocess the data for contextual embeddings
preproc_contextual = PreprocessForContextualEmbeddings()
preproc_contextual.save()

print("Preprocessing completed for contextual embeddings.")

Overwriting preprocess_script.py


In [26]:
cd "/kaggle/working/ML-Project/data"


/kaggle/working/ML-Project/data


In [27]:
mkdir preprocessed

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


mkdir: cannot create directory 'preprocessed': File exists


In [28]:
cd "/kaggle/working/ML-Project"


/kaggle/working/ML-Project


In [29]:
pip install sentence-transformers


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Note: you may need to restart the kernel to use updated packages.


In [30]:
!python preprocess_script.py


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
2024-11-05 16:17:58,086 - logger_config - INFO - Preprocessing for contextual embeddings initialized. (preprocess.py:92)
2024-11-05 16:17:58,213 - logger_config - INFO - Preprocessing for contextual embeddings completed. (preprocess.py:138)
Preprocessing completed for contextual embeddings.


In [31]:
# Cell 2: Generating BERT embeddings
from sentence_transformers import SentenceTransformer
import pandas as pd
import numpy as np
from constants import SAVEPATH

# Load the preprocessed train and test data
train_data_contextual = pd.read_csv(f"{SAVEPATH}/train_contextual_preprocessed.csv")
test_data_contextual = pd.read_csv(f"{SAVEPATH}/test_contextual_preprocessed.csv")

# Load the BERT-based transformer model
model_bert = SentenceTransformer('bert-base-nli-mean-tokens')

# Generate train and test embeddings
train_embeddings_bert = model_bert.encode(train_data_contextual['text'].tolist(), show_progress_bar=True)
test_embeddings_bert = model_bert.encode(test_data_contextual['text'].tolist(), show_progress_bar=True)

# Save the embeddings for future use (optional)
np.save(f"{SAVEPATH}/train_embeddings_bert.npy", train_embeddings_bert)
np.save(f"{SAVEPATH}/test_embeddings_bert.npy", test_embeddings_bert)

print("BERT embeddings generation completed.")


/opt/conda/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Batches:   0%|          | 0/89 [00:00<?, ?it/s]

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

BERT embeddings generation completed.


In [32]:
# Cell 3: Import necessary libraries and prepare data for LSTM
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# Load preprocessed embeddings and labels
train_embeddings = np.load(f"{SAVEPATH}/train_embeddings_bert.npy")
test_embeddings = np.load(f"{SAVEPATH}/test_embeddings_bert.npy")
train_labels = train_data_contextual['label'].values
test_labels = test_data_contextual['label'].values

# Convert data to PyTorch tensors
train_tensor = torch.tensor(train_embeddings, dtype=torch.float32)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)
test_tensor = torch.tensor(test_embeddings, dtype=torch.float32)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)

# Create DataLoader for training and testing
train_dataset = TensorDataset(train_tensor, train_labels_tensor)
test_dataset = TensorDataset(test_tensor, test_labels_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)


In [33]:
# Cell 4: Define the LSTM model class
class LSTMClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        # LSTM expects input of shape (batch_size, seq_length, input_dim)
        # Add an extra dimension to match (batch_size, 1, input_dim)
        x = x.unsqueeze(1)
        _, (hn, _) = self.lstm(x)
        out = self.fc(hn[-1])
        return out

# Hyperparameters
input_dim = train_embeddings.shape[1]  # Number of features from embeddings
hidden_dim = 128
output_dim = len(set(train_labels))  # Number of classes (e.g., 2 for binary classification)
num_layers = 1
num_epochs = 10
learning_rate = 0.001

# Instantiate the model, define the loss function and optimizer
model = LSTMClassifier(input_dim, hidden_dim, output_dim, num_layers)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)


In [34]:
# Cell 5: Train the LSTM model
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for batch_data, batch_labels in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_data)
        loss = criterion(outputs, batch_labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {total_loss/len(train_loader):.4f}")


Epoch [1/10], Loss: 0.4940
Epoch [2/10], Loss: 0.4301
Epoch [3/10], Loss: 0.4032
Epoch [4/10], Loss: 0.3732
Epoch [5/10], Loss: 0.3594
Epoch [6/10], Loss: 0.3479
Epoch [7/10], Loss: 0.3067
Epoch [8/10], Loss: 0.2887
Epoch [9/10], Loss: 0.2692
Epoch [10/10], Loss: 0.2355


In [35]:
# Cell 6: Evaluate the model on both train and test sets
from sklearn.metrics import classification_report

# Function to evaluate the model on a given dataset
def evaluate_model(data_loader, dataset_type="Test"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch_data, batch_labels in data_loader:
            outputs = model(batch_data)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.numpy())
            all_labels.extend(batch_labels.numpy())

    # Calculate and print evaluation metrics
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')
    accuracy = accuracy_score(all_labels, all_preds)

    print(f"{dataset_type} Set Metrics:")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  Accuracy: {accuracy:.4f}")
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds))

# Evaluate on the train set
evaluate_model(train_loader, dataset_type="Train")

# Evaluate on the test set
evaluate_model(test_loader, dataset_type="Test")


Train Set Metrics:
  Precision: 0.9315
  Recall: 0.9306
  F1 Score: 0.9306
  Accuracy: 0.9306

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      1350
           1       0.95      0.91      0.93      1488

    accuracy                           0.93      2838
   macro avg       0.93      0.93      0.93      2838
weighted avg       0.93      0.93      0.93      2838

Test Set Metrics:
  Precision: 0.7523
  Recall: 0.7510
  F1 Score: 0.7511
  Accuracy: 0.7510

Classification Report:
              precision    recall  f1-score   support

           0       0.73      0.77      0.75       346
           1       0.77      0.73      0.75       369

    accuracy                           0.75       715
   macro avg       0.75      0.75      0.75       715
weighted avg       0.75      0.75      0.75       715

